# UNetSR web demo: GPU backend

Runs `demo/server.py` on this Colab GPU and publishes it through ngrok for the web page at https://unetsr-upscaler.vercel.app.

1. The runtime should be **T4 GPU** (Runtime → Change runtime type). This notebook asks for it by default.
2. 🔑 **Secrets** (left sidebar): add `NGROK_AUTHTOKEN` from [dashboard.ngrok.com → Your Authtoken](https://dashboard.ngrok.com/get-started/your-authtoken) (a free account works), with notebook access on.
3. **Runtime → Run all.** Leave the last cell running and open the site link that cell 2 prints. Stop the cell (or the runtime) to take the backend down.

In [ ]:
BRANCH = "master"
REPO = "https://github.com/Rah-Rah-Mitra/UNet-Training-Blended-PBL-"
DOMAIN = "evolved-oarfish-guiding.ngrok-free.app"
SITE = "https://unetsr-upscaler.vercel.app"

!git clone -q --depth 1 -b {BRANCH} {REPO} repo || git -C repo pull -q
for s in (2, 4, 8):
    !wget -q -nc -P repo/demo/weights {REPO}/releases/download/demo-weights-v1/unetsr_x{s}.pt
!pip -q install pyngrok

In [ ]:
from google.colab import userdata
from pyngrok import ngrok

ngrok.kill()  # drop a tunnel left over from an earlier run of this cell
ngrok.set_auth_token(userdata.get("NGROK_AUTHTOKEN"))
try:
    url = ngrok.connect(8765, domain=DOMAIN).public_url
except Exception:  # DOMAIN belongs to the repo owner's ngrok account; any other token gets its own URL
    url = ngrok.connect(8765).public_url
print("backend:", url)
print("open the site:", SITE if DOMAIN in url else f"{SITE}/?api={url}")

In [ ]:
!python repo/demo/server.py